## Model predictions
Import libraries

In [ ]:
import os
import json
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from silx.io.dictdump import h5todict

Define inputs

In [ ]:
model_name = "meno_f32"

In [ ]:
# Models
models = {
  "meno_f32": "./../meno/03.merged/case.r01.f32/saving/noneq/SavedModel/",
  "meno_f64": "./../meno/03.merged/case.r01.f64/saving/noneq/SavedModel/",
  "meno_f64_nocorr": "./../meno/02.dist/case.r01.nocorr/saving/noneq_dist/SavedModel/",
  "deeponet": "./../deeponet/02.merged/case.r04.siso/saving/noneq/SavedModel"
}
# Paths
path_to_datagen = "./../datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": models[model_name],
  "indices": path_to_datagen + "/data/indices.json",
  "inp_space": path_to_datagen + "/datagen/runs/samples/inp_space.hdf5",
  "saving": f"./output/visual/{model_name}"
}
os.makedirs(paths["saving"], exist_ok=True)
# Cases
cases = [5, 9, 19, 40, 43, 54, 57, 65, 78, 99]
# Cases
inp_space_lbl = ["X_O", "p", "Ti", "T"]

In [ ]:
# Time grid
tlim = [1e-11, 1e-1]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Define data indices

In [ ]:
# Data indices
with open(path_to_datagen + "/data/indices.json") as file:
  _indices = json.load(file)
if (model_name in ("meno_f32", "meno_f64")):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"]
    ],
    "y": [
      _indices["na"],
      _indices["nmi"],
      _indices["T"]
    ]
  }
elif (model_name == "deeponet"):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"]
    ],
    "y": [
      _indices["na"] + _indices["nmi"],
      _indices["T"]
    ]
  }
elif (model_name == "meno_f64_nocorr"):
  indices = {
    "x": [
      _indices["na_0"],
      _indices["nmi_0"],
      _indices["T_0"],
      _indices["t"],
      _indices["qoi_eq"],
      _indices["na"]+_indices["T"]
    ],
    "y": [
      _indices["nmi"]
    ]
  }
else:
  raise ValueError()

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])
inp_space = h5todict(paths["inp_space"])["test"]

Plot true vs. predicted solutions

In [ ]:
xinp = []
for (icase, idata) in data.items():
  idata = idata.to_numpy()
  case_nb = int(icase.split("_")[-1])
  if ((cases is not None) and (case_nb not in cases)):
    continue
  xinp.append(inp_space[case_nb-1])
  _, y_true, y_pred = utils.predict(model, idata, indices, t, t_index=3)
  for yscale in ("log", "linear"):
    utils.plot_test(
      path=paths["saving"],
      x=t,
      y=y_true[:,:-1] if (model_name != "meno_f64_nocorr") else y_true,
      y_pred=y_pred[:,:-1] if (model_name != "meno_f64_nocorr") else y_pred,
      figname=icase+"_species_"+yscale,
      figext=["pdf", "pgf"],
      labels=[r"$t$ [s]", r"$n$ [m$^{-3}$]"],
      scales=["log", yscale],
      xlim=tlim,
      save=True,
      show=False
    )
  if (model_name != "meno_f64_nocorr"):
    utils.plot_test(
      path=paths["saving"],
      x=t,
      y=y_true[:,-1:],
      y_pred=y_pred[:,-1:],
      figname=icase+"_temp",
      figext=["pdf", "pgf"],
      labels=[r"$t$ [s]", r"$T$ [K]"],
      scales=["log", "linear"],
      xlim=tlim,
      save=True,
      show=False
    )

In [ ]:
xinp_df = pd.DataFrame(np.vstack(xinp), columns=inp_space_lbl, index=cases)
xinp_df.to_csv(paths["saving"]+"/inp_space.csv", float_format="%.6e")